In [ ]:
import re
import gc
import logging
import random
import json
import polars as pl
import pandas as pd
import time
import requests
import glob
import joblib

from time import sleep
from datetime import datetime
from tqdm.auto import tqdm
from youtube_transcript_api import (
    YouTubeTranscriptApi, 
    NoTranscriptFound, 
    TranscriptsDisabled,
)
from concurrent.futures import ThreadPoolExecutor, as_completed

# pl.Config.set_fmt_str_lengths(1_000)
# pl.Config.set_tbl_width_chars(1_000)

pd.options.display.max_colwidth = 300

In [ ]:
def get_current_datetime():
    """Get current date and time"""
    now = datetime.now()

    # Format the date-time string
    formatted_datetime = now.strftime("%Y-%m-%d_%H_%M_%S")

    return formatted_datetime


def setup_logger(
    prefix: str,
    console_level: str = "DEBUG",
    file_level: str = "WARNING",
):
    """
    Sets up a logger with a console and file handler.
    """
    logger = logging.getLogger(__name__)
    logger.setLevel(
        logging.DEBUG
    )  # Set to the highest level; handlers will filter appropriately

    # Create console handler
    console_handler = logging.StreamHandler()
    console_handler.setLevel(console_level)

    log_file = f"../log/{prefix}_{get_current_datetime()}.log"

    file_handler = logging.FileHandler(log_file, mode="a", encoding="utf-8")
    file_handler.setLevel(file_level)

    # Define log format
    formatter = logging.Formatter(
        "{asctime},{levelname},{message}",
        style="{",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    # Assign formatter to handlers
    console_handler.setFormatter(formatter)
    file_handler.setFormatter(formatter)

    # Add handlers to logger
    logger.addHandler(console_handler)
    logger.addHandler(file_handler)

    return logger

In [ ]:
from datetime import datetime

# Get today's date
today_date = datetime.now()

# Format it as a string (e.g., "YYYY-MM-DD")
today_string = today_date.strftime("%Y-%m-%d").replace("-", "")

print(today_string, type(today_string))

In [ ]:
# done data
path = [
    "../data/channel_df_20241212_snap_00.csv",
    "../data/chan`nel_df_20241212_snap_01.csv"
]
dfs = []
for p in path:
    d = pl.read_csv(p)
    dfs.append(d)

previous = pl.concat(dfs, how="vertical").unique(subset="video_id")
previous

In [ ]:
# current data
path = "../data/channel_df_20241212.csv"
video_df = pl.read_csv(path)
video_df.shape

In [ ]:
video_df

In [ ]:
# filter out what is done
video_df = video_df.filter(
    pl.col("video_id").is_in(previous["video_id"]).not_()
)
video_df.shape

In [ ]:
video_df.sample(3)

In [ ]:
# https://free-proxy-list.net/anonymous-proxy.html
path = "./proxies_20250227.txt"
with open(path, "r") as f:
    p_list = f.read().splitlines()

proxies1 = [ {"http" : "http://" + p} for p in p_list]
proxies1

In [ ]:
proxies2 = [
    # "http://171.4.196.128:8080",
    # "socks4://202.29.218.138:4153",
    # "http://203.144.144.146:8080",
    # "http://49.49.38.127:8080",
    # "http://171.99.252.63:8080",
    # "socks4://1.179.148.9:36476",
    # "http://58.64.12.11:8081",
    # "http://103.253.72.220:8080",
    # "http://49.48.68.93:8080",
    # "http://171.6.9.175:8080",
    # "http://27.254.99.183:8118",
    # "http://49.229.100.235:8080",
    # "http://110.164.233.42:8080",
    # "socks4://8.213.222.157:3128",
    # "socks4://159.192.139.42:5678"
    "http://171.4.185.208:8080",
    "http://122.154.75.204:8080",
    "http://49.48.103.194:8080",
    "http://58.136.169.187:8080",
    "http://183.88.224.86:8080",
    "http://49.0.87.62:8088",
    "http://203.150.172.151:8080",
    "http://110.49.53.69:8081",
    
]

proxies2 = [
    {"http" : p} if "http" in p else {"socks4" : p} for p in proxies2
]
# random.choice(proxies)
proxies = proxies1 + proxies2

## one sample

In [ ]:
transcript_list = YouTubeTranscriptApi.list_transcripts(
    "9nDjgusMmCY", 
    proxies=proxies[-1]
)
transcript = transcript_list.find_transcript(["th"])

In [ ]:
for i in transcript_list:
    print(i)

In [ ]:
type(transcript.fetch())

In [ ]:
transcript.fetch()

In [ ]:
transcript.is_generated

## Call API

In [ ]:
# def save_to_json(
#     subtitles: list[dict], 
#     id:str, 
#     file_path_json: str,
# ):
#     """
#     Append data to a JSON file in a thread-safe manner.
#     """
#     try:
#         # Read existing data
#         with open(file_path_json, "r") as file:
#             existing_data = json.load(file)
#     except FileNotFoundError:
#         # Start with an empty list if the file does not exist
#         existing_data = []

#     # Add new data to the existing data
#     # existing_data.extend(subtitles)
#     existing_data.append(subtitles)

#     # Write the updated data back to the file
#     with open(file_path_json, "w") as file:
#         json.dump(existing_data, file, indent=4, ensure_ascii=False)

In [ ]:
ids = video_df["video_id"].unique().cast(pl.String).to_list()[20:30]

In [ ]:
file_path_json = f"../data/subtitles_{today_string}.jsonl"
# file_path_csv = f"../data/subtitles_{today_string}.csv"
sleep_min = 3
sleep_max = 8

items = []

for id in tqdm(ids):
    proxies_tried = set()
    random_proxy = random.choice(proxies)
    while len(proxies_tried) < len(proxies):
        proxy = random.choice(
            [p for p in proxies if tuple(p.items()) not in proxies_tried]
        )
        proxies_tried.add(tuple(proxy.items()))

        try:
            transcript_list = YouTubeTranscriptApi.list_transcripts(
                id, proxies=random_proxy
            )
            transcript = transcript_list.find_transcript(["th"])
            subtitles = transcript.fetch()

        except NoTranscriptFound:
            logger.error(f"There is no th subtitle for {id}")
            sleep(random.randint(sleep_min, sleep_max))
            break
        except TranscriptsDisabled:
            logger.error(f"Subtitle is disabled for {id}")
            sleep(random.randint(sleep_min, sleep_max))
            break
        except requests.exceptions.ProxyError:
            logger.warning(
                f"Proxy {random_proxy} can not be used for {id}. Try another..."
            )
            sleep(random.randint(sleep_min, sleep_max))
            continue

        if subtitles:
            for order, sub in enumerate(subtitles):
                sub["order"] = order + 1
                sub["id"] = transcript.video_id
                sub["is_generated"] = transcript.is_generated
            # save_to_json(subtitles, file_path_json)
            pd.DataFrame(subtitles).to_json(
                file_path_json,
                mode="a",
                index=False,
                orient="records",
                force_ascii=False,
                lines=True,
            )
            # items.append(subtitles)
            logger.info(f"finish video {id}")
            # sleep(random.randint(sleep_min, sleep_max))
            break
        # sleep(random.randint(sleep_min, sleep_max))

# Filter license

In [ ]:
def get_current_datetime():
    """Get current date and time"""
    now = datetime.now()

    # Format the date-time string
    formatted_datetime = now.strftime("%Y-%m-%d_%H_%M_%S")

    return formatted_datetime


def setup_logger(
    prefix: str,
    console_level: str = "DEBUG",
    file_level: str = "WARNING",
):
    """
    Sets up a logger with a console and file handler.
    """
    logger = logging.getLogger(__name__)
    logger.setLevel(
        logging.DEBUG
    )  # Set to the highest level; handlers will filter appropriately

    # Create console handler
    console_handler = logging.StreamHandler()
    console_handler.setLevel(console_level)

    log_file = f"../log/{prefix}_{get_current_datetime()}.log"

    file_handler = logging.FileHandler(log_file, mode="a", encoding="utf-8")
    file_handler.setLevel(file_level)

    # Define log format
    formatter = logging.Formatter(
        "{asctime},{levelname},{message}",
        style="{",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    # Assign formatter to handlers
    console_handler.setFormatter(formatter)
    file_handler.setFormatter(formatter)

    # Add handlers to logger
    logger.addHandler(console_handler)
    logger.addHandler(file_handler)

    return logger

In [ ]:
video_df = pd.read_csv("../data/channel_df_20241212.csv"); video_df.info()
video_df = video_df.drop_duplicates(subset="video_id").dropna(subset="video_id")

In [ ]:
query = video_df["query"].str.strip('""').str.replace(" ", "", regex=False).str.lower()
channel = video_df["channel"].str.strip('""').str.replace(" ", "", regex=False).str.lower()

In [ ]:
filter_ = query.eq(channel)

In [ ]:
video_df = video_df.loc[filter_]

In [ ]:
video_df.shape

In [ ]:
video_df.sample(2)

In [ ]:
# DEMO
import json
import yt_dlp

# help(yt_dlp.YoutubeDL)

In [ ]:
# # https://free-proxy-list.net/anonymous-proxy.html
path = "./proxies_20250227.txt"
with open(path, "r") as f:
    p_list = f.read().splitlines()
    
# proxies = ["http://" + p for p in p_list]
# proxies
p_list

In [ ]:
# DEMO
import json
import yt_dlp

URL = "https://www.youtube.com/watch?v=hbge2TdiO6w"

# ℹ️ See help(yt_dlp.YoutubeDL) for a list of available options and public functions
ydl_opts = {
    "quiet" : True,
    "skip_download" : True,
    "socket_timeout" : 15,
    "extractor_retries" : 5,
    # "proxy" : "http://50.174.7.156:80"
}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info("ytsearch1:hbge2TdiO6w", download=False)
    # info = json.dumps(ydl.sanitize_info(info))

    # ℹ️ ydl.sanitize_info makes the info json-serializable
    # print(json.dumps(ydl.sanitize_info(info)))

In [ ]:
info

In [ ]:
NO_LICENSE = "NO_LICENSE"
def check_licnese_and_sub(video_url:str, ydl_opts_confg:dict):
    with yt_dlp.YoutubeDL(ydl_opts_confg) as ydl:
        info = ydl.extract_info(video_url, download=False)
    license = info["license"] if info["license"] else NO_LICENSE
    has_sub = True if bool(info["subtitles"]) or bool(info["automatic_captions"]) else False
    return {
        "video_url" : video_url, "license" : license, "has_sub" : has_sub
    }

In [ ]:
ydl_opts = {
    "quiet" : True,
    "skip_download" : True,
    "socket_timeout" : 15,
    "extractor_retries" : 5,
    # "proxy" : "http://50.174.7.156:80"
}
check_licnese_and_sub(video_url="https://www.youtube.com/watch?v=hbge2TdiO6w", 
                      ydl_opts_confg=ydl_opts)

In [ ]:
logger = setup_logger("yt_check_license")
check_license_list = []
ydl_opts = {
    "quiet" : True,
    "skip_download" : True,
    "socket_timeout" : 15,
    "extractor_retries" : 5,
}
for video_url in tqdm(video_df["video_url"]):
    # sleep_time = random.randint(8, 120)
    try:
        outout_item = check_licnese_and_sub(video_url, ydl_opts)
        logger.info(f"Succesful get info for {video_url}")
        check_license_list.append(outout_item)
        # sleep(sleep_time)
    except Exception as e:
        logger.error(f"Error with video {video_url}")
        outout_item = {
            "video_url" : video_url, "license" : False, "has_sub" : False
        }
        logger.warning(f"Add dummy get info for {video_url}")
        check_license_list.append(outout_item)
        # sleep(sleep_time)

In [ ]:
check_license_df = pd.DataFrame(check_license_list)

In [ ]:
import json
import yt_dlp
import concurrent.futures
from tqdm.notebook import tqdm

NO_LICENSE = "NO_LICENSE"

def check_license_and_sub(video_url: str, ydl_opts_config: dict):
    """
    Extracts video information using yt_dlp and returns the license and subtitle status.
    """
    with yt_dlp.YoutubeDL(ydl_opts_config) as ydl:
        info = ydl.extract_info(video_url, download=False)
    license_val = info.get("license") if info.get("license") else NO_LICENSE
    has_sub = bool(info.get("subtitles")) or bool(info.get("automatic_captions"))
    return {"video_url": video_url, "license": license_val, "has_sub": has_sub}

def process_video(video_url):
    """
    Wrapper function that handles exceptions during info extraction.
    Returns a dummy dictionary if an error occurs.
    """
    try:
        result = check_license_and_sub(video_url, ydl_opts)
        logger.info(f"Successfully got info for {video_url}")
    except Exception as e:
        logger.error(f"Error with video {video_url}: {e}")
        result = {"video_url": video_url, "license": False, "has_sub": False}
        logger.warning(f"Added dummy info for {video_url}")
    return result

# Assuming setup_logger is defined elsewhere in your code
logger = setup_logger("yt_check_license")

# yt_dlp options
ydl_opts = {
    "quiet": True,
    "skip_download": True,
    "socket_timeout": 15,
    "extractor_retries": 5,
}

# Convert the DataFrame column to a list of URLs
video_urls = video_df["video_url"].tolist()

In [ ]:
# Use ThreadPoolExecutor to process videos concurrently
check_license_list = []
with concurrent.futures.ThreadPoolExecutor(max_workers=128) as executor:
    results = list(tqdm(executor.map(process_video, video_urls), total=len(video_urls)))
    check_license_list.extend(results)

# Multi Thread

In [ ]:
import threading

json_lock = threading.Lock()


def save_to_json(subtitles: list[dict], file_path_json: str):
    """
    Append a list of dictionaries (subtitles) to a JSON file in a thread-safe manner using pandas.
    """
    with json_lock:  # Ensure only one thread writes to the file at a time
        pd.DataFrame(subtitles).to_json(
            file_path_json,
            mode="a",
            index=False,
            orient="records",
            force_ascii=False,
            lines=True,
        )


def fetch_and_save_subtitles(video_id, proxies, file_path_json, sleep_min, sleep_max):
    """
    Fetch subtitles for a single video and save them to the JSON file.
    """
    proxies_tried = set()
    # random_proxy = random.choice(proxies)

    while len(proxies_tried) < len(proxies):
        proxy = random.choice(
            [p for p in proxies if tuple(p.items()) not in proxies_tried]
        )
        proxies_tried.add(tuple(proxy.items()))

        try:
            transcript_list = YouTubeTranscriptApi.list_transcripts(
                video_id,
                proxies=proxy,
            )
            transcript = transcript_list.find_transcript(["th"])
            subtitles = transcript.fetch()

            # If subtitles are found, process and save them
            if subtitles:
                for order, sub in enumerate(subtitles):
                    sub["order"] = order + 1
                    sub["id"] = transcript.video_id
                    sub["is_generated"] = transcript.is_generated
                save_to_json(subtitles, file_path_json)
                logger.info(f"Finished video {video_id}")
            time.sleep(random.randint(sleep_min, sleep_max))
            return

        except NoTranscriptFound:
            logger.error(f"No 'th' subtitles for video {video_id}")
            break
        except TranscriptsDisabled:
            logger.error(f"Subtitles are disabled for video {video_id}")
            break
        except requests.exceptions.ProxyError:
            logger.warning(f"Proxy {proxy} failed for video {video_id}. Retrying...")
            continue
        except Exception as e:
            logger.error(f"There is an error getting subtitle for {id}", exc_info=True)

    time.sleep(random.randint(sleep_min, sleep_max))

In [ ]:
ids = video_df["video_id"].unique().cast(pl.String).to_list()
file_path_json = f"../data/subtitles_{today_string}.json"
sleep_min = 3
sleep_max = 8
max_threads = 64  # Adjust based on the number of threads your system can handle
logger = setup_logger("yt_get_subtitle")

In [ ]:
with ThreadPoolExecutor(max_threads) as executor:
    futures = [
        executor.submit(
            fetch_and_save_subtitles,
            video_id,
            proxies,
            file_path_json,
            sleep_min,
            sleep_max,
        )
        for video_id in ids
    ]

    # Track progress with tqdm
    for future in tqdm(
        as_completed(futures), total=len(futures), desc="Processing Videos"
    ):
        try:
            future.result()  # Raise exceptions if any occurred
        except Exception as e:
            logger.error(f"Error processing a video: {e}", exc_info=True)

In [ ]:
1

# Get subtitle

In [ ]:
# Sample DataFrame
# data = {
#     "text": [
#         "การวาดและระบายสี",
#         "ในโปรแกรม GIMP",
#         "มีเครื่องมือวาดและระบายสีอยู่หลายตัวด้วยกันค่ะ",
#         "สามารถเลือกเครื่องมือวาดและระบายสีได้จาก Toolbox",
#         "หรือว่าเลือกจาก Image Menu",
#         "เลือกแถบ Toos และเลือก Pain Tools",
#         "คุณสมบัติพื้นฐานของเครื่องมือวาดและระบายสี",
#         "เครื่องมือในกลุ่มวาดและระบายสี",
#         "มีคุณสมบัติพื้นฐานใน Tool Options",
#         "เหมือนๆกันอยู่กลุ่มหนึ่งค่ะ",
#         "ได้แก่ Mode",
#         "ใช้เลือกโหมดการระบายหรือโหมดการซ้อนทับกับภาพด้านล่าง",
#         "โดยปกติ เมื่อระบายอะไรลงไปบนภาพ",
#         "จะเป็นการระบายทับ",
#         "โหมด Nomal",
#     ],
#     "start": [23.76, 26.0, 27.1, 31.0, 35.04, 38.5, 52.0, 57.0, 59.04, 63.02, 65.0, 67.76, 73.06, 76.72, 79.84],
#     "end": [26.0, 27.08, 31.0, 35.0, 38.1, 41.88, 56.04, 59.02, 63.02, 65.0, 67.72, 73.02, 76.64, 79.84, 81.04],
# }

# df = pd.DataFrame(data)
# display(df)
# # Process the subtitles with the specified gap condition
# processed_text = add_space_between_thai_and_english(
#     concatenate_subtitles(df)
# )
# # Output the processed text
# print(processed_text)

In [ ]:
# # Function to concatenate subtitles based on the time gap condition
# def concatenate_subtitles(df, max_gap=1):
#     result = []
#     prev_end = None

#     for _, row in df.iterrows():
#         text, start, end = row["text"], row["start"], row["end"]

#         # Check if the gap between subtitles exceeds the max allowed gap
#         if (prev_end is not None) and (start - prev_end > max_gap):
#             result.append("\n")  # Add a newline if the gap is larger than allowed

#         result.append(text)
#         prev_end = end

#     return "".join(result)

# # Function to add spaces between Thai and English characters
# def add_space_between_thai_and_english(text):
#     # Add space between Thai characters and English characters
#     text = re.sub(r"([ก-๙])([a-zA-Z0-9])", r"\1 \2", text)
#     text = re.sub(r"([a-zA-Z0-9])([ก-๙])", r"\1 \2", text)
#     return text

In [ ]:
PAT1 = re.compile(r"([ก-๙])([a-zA-Z0-9])")
PAT2 = re.compile(r"([a-zA-Z0-9])([ก-๙])")

In [ ]:
def concatenate_subtitles(df: pd.DataFrame, max_gap: int = 1) -> str:
    result = []
    prev_end = None

    for start, end, text in zip(df["start"], df["end"], df["text"]):
        if prev_end is not None and (start - prev_end > max_gap):
            result.append("\n")
        result.append(text)
        prev_end = end

    return "".join(result)

def add_space_between_thai_and_english(text: str) -> str:
    # Add space between Thai characters and English characters
    text = PAT1.sub(r"\1 \2", text)
    text = PAT2.sub(r"\1 \2", text)
    return text

# Optimized processing loop
def process_subtitles(sub_df: pd.DataFrame) -> list[dict]:
    json_list = []

    # Pre-group the DataFrame by "id" to avoid multiple calls to `unique()`
    grouped = sub_df.groupby("id")

    for id, video_sub_df in tqdm(grouped):
        # Convert to a dictionary for faster access and processing
        video_sub_dict = video_sub_df.to_dict(orient="list")

        # Concatenate subtitles and process text
        processed_text = add_space_between_thai_and_english(
            concatenate_subtitles(pd.DataFrame(video_sub_dict))
        )

        # Prepare the item
        item = {
            "id": id,
            "text": processed_text,
            "source": "youtube",
            "metadata": {
                "license": "Creative Commons Attribution license (reuse allowed)",
                "channel_name": video_sub_dict["channel"][0],
                "title": video_sub_dict["title"][0],
                "video_url": video_sub_dict["video_url"][0],
                "is_subtitle_generated": str(video_sub_dict["is_generated"][0]),
            },
        }

        json_list.append(item)

        # Explicit garbage collection
        del video_sub_dict
        gc.collect()

    return json_list

In [ ]:
json_subtitles_path = glob.glob("../data/subtitles_*.jsonl")
dfs = [pd.read_json(path, lines=True) for path in json_subtitles_path]

In [ ]:
json_subtitles_path

In [ ]:
for df in dfs: print(df.shape)

In [ ]:
assert set(dfs[0]["id"]).intersection(set(dfs[1]["id"])) == set()
assert set(dfs[0]["id"]).intersection(set(dfs[2]["id"])) == set()
assert set(dfs[1]["id"]).intersection(set(dfs[2]["id"])) == set()

In [ ]:
sub_df = pd.concat(dfs)
sub_df.shape

In [ ]:
sub_df.head(3)

In [ ]:
sub_df["id"].nunique()

In [ ]:
path = "../data/channel_df_20241212.csv"
video_df = (
    pd.read_csv(path)
    .rename(columns={"video_id": "id"})
    .loc[:, ["id", "title", "channel", "video_url"]]
    .drop_duplicates(subset="id")
    .dropna(subset="id")
)
video_df.shape

In [ ]:
sub_df = sub_df.merge(video_df, on="id", how="left")
sub_df.shape

In [ ]:
sub_df.head(2)

In [ ]:
sub_df.isnull().sum()

In [ ]:
sub_df.shape, sub_df["video_url"].nunique()

## Filter license

In [ ]:
import yt_dlp

In [ ]:
NO_LICENSE = "NO_LICENSE"
def check_licnese(video_url:str, ydl_opts_confg:dict):
    with yt_dlp.YoutubeDL(ydl_opts_confg) as ydl:
        info = ydl.extract_info(video_url, download=False)
    license = info["license"] if info["license"] else NO_LICENSE
    return {"video_url" : video_url, "license" : license}

In [ ]:
ydl_opts = {
    "quiet" : True,
    "skip_download" : True,
    "socket_timeout" : 15,
    "extractor_retries" : 5,
    # "proxy" : "https://47.238.67.96:8888/"
}
test_url = "https://www.youtube.com/watch?v=1ErWStEOOjI"
out = check_licnese(video_url=test_url, ydl_opts_confg=ydl_opts)

In [ ]:
out

In [ ]:
logger = setup_logger("yt_check_license")
check_license_list = []
ydl_opts = {
    "quiet" : True,
    "skip_download" : True,
    "socket_timeout" : 15,
    "extractor_retries" : 7,
}
for video_url in tqdm(sub_df["video_url"].unique()):
    sleep_time = random.randint(3, 14)
    try:
        out = check_licnese(video_url, ydl_opts)
        logger.info(f"Succesful get info for {video_url}")
        check_license_list.append(out)
        logger.info(f"Sleep for {sleep_time}")
        sleep(sleep_time)
    except Exception as e:
        logger.error(f"Error with video {video_url}")
        out = {"video_url" : video_url, "license" : False}
        logger.warning(f"Add dummy info for {video_url}")
        check_license_list.append(out)
        logger.info(f"Sleep for {sleep_time}")
        sleep(sleep_time)

In [ ]:
import concurrent.futures
import logging
import time
import yt_dlp  # Make sure yt_dlp is installed

logger = setup_logger("yt_check_license")

# Function to fetch license info for a single video URL with retries
def fetch_license(url, max_retries=3):
    """Fetch the license information for a YouTube video URL."""
    attempt = 0
    while attempt < max_retries:
        try:
            # Use yt_dlp to get video info without downloading the video
            ydl_opts = {"quiet": True}  # suppress yt_dlp output, we handle logging
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                logger.info(f"Processing URL: {url}")  # log start
                info = ydl.extract_info(url, download=False)
                # Extract the license field (if present)
                license_info = info.get("license")
                if not license_info:
                    license_info = "License info not found"
                logger.info(f"Successfully retrieved license for {url}: {license_info}")
                return license_info
        except Exception as e:
            attempt += 1
            # If not the last attempt, log a warning and retry after a short delay
            if attempt < max_retries:
                # Exponential backoff: e.g., 1s, 2s, 4s
                delay = 1 * (2 ** (attempt - 1))
                logger.warning(f"Error processing {url} (attempt {attempt}/{max_retries}): {e}. Retrying in {delay}s...")
                time.sleep(delay)
            else:
                # Last attempt failed, log an error
                logger.error(f"Failed to retrieve license for {url} after {max_retries} attempts. Error: {e}")
                return None

In [ ]:
# Determine number of threads (workers) for concurrency
# num_workers = min(5, len(video_urls))  # e.g., use up to 5 threads or one per URL if fewer URLs
num_workers = 16

# Use ThreadPoolExecutor to process URLs concurrently with a progress bar
results = {}  # Store results as {url: license_info or None}
with concurrent.futures.ThreadPoolExecutor(max_workers=num_workers) as executor:
    # Submit all tasks to the thread pool
    future_to_url = {executor.submit(fetch_license, url): url for url in sub_df["video_url"].unique()}
    # Use tqdm to display a progress bar
    for future in tqdm(
        concurrent.futures.as_completed(future_to_url), 
        total=len(future_to_url), 
        desc="Processing videos"
    ):
        url = future_to_url[future]
        try:
            license_info = future.result()
        except Exception as exc:
            logger.error(f"Unhandled exception for URL {url}: {exc}")
            license_info = None
        results[url] = license_info

In [ ]:
# (Optional) Do something with results, e.g., print summary
print("\nLicense extraction results:")
for url, license_info in results.items():
    status = license_info if license_info is not None else "Failed to retrieve"
    print(f"{url} -> {status}")

In [ ]:
import joblib
joblib.dump(results, "./yt_license.joblib")

In [ ]:
import joblib
results = joblib.load("./yt_license.joblib")

In [ ]:
license_list = []
for k, v in results.items():
    license_dict = {}
    license_dict["video_url"] = k
    license_dict["license"] = v
    license_list.append(license_dict)

In [ ]:
sub_df["video_url"].nunique()

In [ ]:
license_df = pd.DataFrame(license_list)
license_df["license"].value_counts(dropna=False)

In [ ]:
video_with_license = license_df.dropna(subset="license").loc[license_df["license"].ne("License info not found")]["video_url"].to_list()
len(video_with_license)

In [ ]:
# **** ONLY LICENSE video ***
sub_df = sub_df.loc[sub_df["video_url"].isin(video_with_license)]

In [ ]:
sub_df["video_url"].nunique()

In [ ]:
sub_df.head(2)

In [ ]:
id_sub_gen = sub_df.loc[:, ["id", "is_generated"]].drop_duplicates()
a = id_sub_gen["is_generated"].value_counts()
b = id_sub_gen["is_generated"].value_counts(normalize=True).mul(100)
c = pd.concat([a, b], axis=1)
c

In [ ]:
sub_id = id_sub_gen.loc[id_sub_gen["is_generated"].eq(False), "id"]
sub_gen_id = id_sub_gen.loc[id_sub_gen["is_generated"].eq(True), "id"]

sub_id_df = sub_df.loc[sub_df["id"].isin(sub_id)].sort_values(by=["id", "order"])
sub_gen_id_df = sub_df.loc[sub_df["id"].isin(sub_gen_id)].sort_values(
    by=["id", "order"]
)

## sub แบบ ไม่ gen

In [ ]:
sub_id_df["end"] = sub_id_df["start"] + sub_id_df["duration"]

In [ ]:
sub_id_df["id"].nunique()

In [ ]:
json_list = process_subtitles(sub_id_df)

In [ ]:
pd.DataFrame(json_list).to_json(
    f"../data/youtube_extracted_subtitles_notgen_{today_string}.jsonl", 
    orient="records", 
    lines=True,
    force_ascii=False
)

## sub แบบ gen

In [ ]:
sub_gen_id_df["end"] = sub_gen_id_df["start"] + sub_gen_id_df["duration"]

In [ ]:
sub_gen_id_df["id"].nunique()

In [ ]:
json_list = process_subtitles(sub_gen_id_df)

In [ ]:
pd.DataFrame(json_list).to_json(
    f"../data/youtube_extracted_subtitles_gen_{today_string}.jsonl", 
    orient="records", 
    lines=True,
    force_ascii=False
)

# tmp